In [81]:
# Importando Bibliotecas
import pandas as pd
import numpy as np
import re

# Carregando a base de dados (especificar separador e encoding para remover BOM e ler corretamente)
# Ler todas as colunas inicialmente como texto para facilitar limpeza
df = pd.read_csv('vendas_nao_tratadas.csv', sep=';', encoding='utf-8-sig', dtype=str)
# Remover BOM (\ufeff) e espaços nos nomes das colunas, caso existam
df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()

## 1. Importação e diagnóstico

Importe pandas, carregue o CSV e examine dimensões, tipos, primeiras linhas, valores ausentes e duplicatas.

In [82]:
# 2. Diagnosticar tipos, nulos e duplicatas.
print("Dimensões:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)
print(f"\nValores ausentes: {df.isnull().sum()}")
print(f"Total de duplicatas: {df.duplicated().sum()}")


Dimensões: (300, 12)

Tipos de dados:
id_venda           str
data_venda         str
cliente            str
email              str
cidade             str
uf                 str
produto            str
categoria          str
quantidade         str
preco_unitario     str
forma_pagamento    str
status_pedido      str
dtype: object

Valores ausentes: id_venda            0
data_venda          0
cliente             0
email               0
cidade              0
uf                  0
produto             0
categoria           0
quantidade         12
preco_unitario      0
forma_pagamento     0
status_pedido       0
dtype: int64
Total de duplicatas: 8


## 2. Padronização textual

Remova espaços extras; padronize cliente em formato de nome, cidade/categoria com capitalização adequada, UF em maiúsculas, forma de pagamento e status.

In [83]:
# 3. Remover espaços e padronizar capitalização.
colunas_texto = df.select_dtypes(include=['object', 'string']).columns
for col in colunas_texto:
    print(col)
    if col in ['cliente', 'cidade', 'categoria', 'produto', 'forma_pagamento']:
        df[col] = df[col].str.strip().str.title()
    elif col == 'uf':
        df[col] = df[col].str.strip().str.upper()
    else:
        df[col] = df[col].str.strip().str.lower()


id_venda
data_venda
cliente
email
cidade
uf
produto
categoria
quantidade
preco_unitario
forma_pagamento
status_pedido


## 3. Conversão de tipos

Converta datas aceitando `AAAA-MM-DD` e `DD/MM/AAAA`; troque vírgula decimal por ponto; converta quantidade e preço para números.

In [84]:
# 4. Converter data, quantidade e preço para os tipos adequados.
# Convertendo as datas
# Criando uma cópia da coluna
s = df['data_venda'].astype(str).str.strip().replace({'nan': None})

# Máscaras para os formatos
mask_iso = s.str.match(r'^\d{4}-\d{2}-\d{2}$')
mask_dmy = s.str.match(r'^\d{1,2}/\d{1,2}/\d{4}$')

# Cria uma lista de datas "nulas" (NaT)
res = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')

# Manda cada grupo para um parser
if mask_iso.any():
    res.loc[mask_iso] = pd.to_datetime(s.loc[mask_iso], format='%Y-%m-%d', errors='coerce')
if mask_dmy.any():
    res.loc[mask_dmy] = pd.to_datetime(s.loc[mask_dmy], format='%d/%m/%Y', errors='coerce')

# Atribui de volta
df['data_venda'] = res

# Remove registros com datas inválidas
qtd_antes = len(df)
df = df[df['data_venda'].notna()]
print(f"Registros removidos por data inválida: {qtd_antes - len(df)}")
print(f"Dimensões após limpeza de datas: {df.shape}")

# Converte a quantidade
df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')

# Remove registros com quantidade inválida
qtd_antes = len(df)
df = df[df['quantidade'].notna()]
print(f"Registros removidos por quantidade inválida: {qtd_antes - len(df)}")
print(f"Dimensões após limpeza de quantidade: {df.shape}")

# Converte o preço unitário
df['preco_unitario'] = df['preco_unitario'].str.replace(',', '.', regex=False)
df['preco_unitario'] = pd.to_numeric(df['preco_unitario'], errors='coerce')

# Remove registros com preço inválido
qtd_antes = len(df)
df = df[df['preco_unitario'].notna()]
print(f"Registros removidos por preço inválido: {qtd_antes - len(df)}")
print(f"Dimensões após limpeza de preço: {df.shape}")

df

Registros removidos por data inválida: 0
Dimensões após limpeza de datas: (300, 12)
Registros removidos por quantidade inválida: 12
Dimensões após limpeza de quantidade: (288, 12)
Registros removidos por preço inválido: 0
Dimensões após limpeza de preço: (288, 12)


,id_venda,data_venda,cliente,email,cidade,uf,produto,categoria,quantidade,preco_unitario,forma_pagamento,status_pedido
1,v0002,2025-01-15,Patrícia Nunes,patricia.nunes@email.com,Fortaleza,CE,Teclado Mecânico,Acessórios,3.0,389.5,Boleto,concluído
2,v0003,2025-01-22,Bruno Lima,bruno.lima@email.com,Campinas,SP,Webcam,Eletrônicos,4.0,319.0,Cartão De Débito,pendente
3,v0004,2025-01-29,Isabela Ribeiro,isabela.ribeiro@email.com,Belo Horizonte,MG,Cadeira Ergonômica,Móveis,5.0,1490.0,Pix,cancelado
4,v0005,2025-02-05,Rafael Teixeira,rafael.teixeira@email.com,Recife,PE,Mouse Sem Fio,Acessórios,1.0,129.9,Cartão De Crédito,concluído
5,v0006,2025-02-12,Carla Mendes,carla.mendes@email.com,Goiânia,GO,Headset,Acessórios,2.0,279.9,Boleto,concluído
...,...,...,...,...,...,...,...,...,...,...,...,...
294,v0053,2025-02-11,Lucas Ferreira,lucas.ferreira@email.com,Campinas,SP,Mouse Sem Fio,Acessórios,4.0,129.9,Cartão De Crédito,pendente
296,v0112,2025-05-05,Eduarda Alves,eduarda.alves@email.com,Fortaleza,CE,Notebook,Eletrônicos,3.0,3499.9,Pix,concluído
297,v0157,2025-04-20,William Carvalho,william.carvalho@email.com,Rio De Janeiro,RJ,Mouse Sem Fio,Acessórios,3.0,129.9,Cartão De Crédito,concluído
298,v0211,2025-06-07,Thiago Moreira,thiago.moreira@email.com,Curitiba,PR,Webcam,Eletrônicos,2.0,319.0,Cartão De Débito,concluído


## 4. Validação

Identifique e-mails inválidos, quantidades/preços ausentes ou não positivos e datas inválidas. Decida como tratar e documente.

In [85]:
# 5. Validar e-mails, datas e valores positivos.

# e-mails
def validar_email(email):
    if pd.isna(email):
        return False
    padrao = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(padrao, str(email)))

linhas_invalidas = pd.DataFrame()

emails_invalidos = ~df['email'].apply(validar_email)
print(f"E-mails inválidos: {emails_invalidos.sum()}")

# quantidades
quantidade_invalida = (df['quantidade'].isna()) | (df['quantidade'] <= 0)
print(f"Quantidades ausentes ou não positivas: {quantidade_invalida.sum()}")

# preço
preco_invalido = (df['preco_unitario'].isna()) | (df['preco_unitario'] <= 0)
print(f"Preços ausentes ou não positivos: {preco_invalido.sum()}")

# datas
data_invalida = df['data_venda'].isna()
print(f"Datas inválidas: {data_invalida.sum()}")

# Removendo tudo
df_valido = df[~(emails_invalidos | quantidade_invalida | preco_invalido | data_invalida)].copy()
print(f"\nLinhas válidas: {len(df_valido)}")


E-mails inválidos: 9
Quantidades ausentes ou não positivas: 0
Preços ausentes ou não positivos: 0
Datas inválidas: 0

Linhas válidas: 279


## 5. Duplicatas e organização

Remova duplicatas exatas, ordene por data e id, redefina o índice e deixe as colunas na ordem original.

In [86]:
# 6. Remover linhas inválidas e duplicatas exatas.
df_valido = df_valido.drop_duplicates().reset_index(drop=True)

df = df_valido
print(f"\nDimensões finais: {df_valido.shape}")
print("\nPrimeiras linhas após limpeza e organização:")
print(df.head())



Dimensões finais: (272, 12)

Primeiras linhas após limpeza e organização:
  id_venda data_venda          cliente                      email  \
0    v0002 2025-01-15   Patrícia Nunes   patricia.nunes@email.com   
1    v0003 2025-01-22       Bruno Lima       bruno.lima@email.com   
2    v0004 2025-01-29  Isabela Ribeiro  isabela.ribeiro@email.com   
3    v0005 2025-02-05  Rafael Teixeira  rafael.teixeira@email.com   
4    v0006 2025-02-12     Carla Mendes     carla.mendes@email.com   

           cidade  uf             produto    categoria  quantidade  \
0       Fortaleza  CE    Teclado Mecânico   Acessórios         3.0   
1        Campinas  SP              Webcam  Eletrônicos         4.0   
2  Belo Horizonte  MG  Cadeira Ergonômica       Móveis         5.0   
3          Recife  PE       Mouse Sem Fio   Acessórios         1.0   
4         Goiânia  GO             Headset   Acessórios         2.0   

   preco_unitario    forma_pagamento status_pedido  
0           389.5             Boleto

In [87]:
# 7. Ordenar por data_venda e id_venda, redefinir o índice.
df = df.sort_values(by=['data_venda', 'id_venda']).reset_index(drop=True)
print("Dataframe ordenado e índice redefinido!")
df


Dataframe ordenado e índice redefinido!


,id_venda,data_venda,cliente,email,cidade,uf,produto,categoria,quantidade,preco_unitario,forma_pagamento,status_pedido
0,v0283,2025-01-02,Bruno Lima,bruno.lima@email.com,Campinas,SP,Webcam,Eletrônicos,4.0,319.0,Cartão De Débito,concluido
1,v0236,2025-01-03,Mariana Oliveira,mariana.oliveira@email.com,Goiânia,GO,Cadeira Ergonômica,Móveis,2.0,1490.0,Pix,concluido
2,v0189,2025-01-04,Daniel Rocha,daniel.rocha@email.com,Porto Alegre,RS,Mouse Sem Fio,Acessórios,5.0,129.9,Cartão De Crédito,concluido
3,v0142,2025-01-05,Patrícia Nunes,patricia.nunes@email.com,Fortaleza,CE,Headset,Acessórios,3.0,279.9,Boleto,concluido
4,v0095,2025-01-06,Felipe Santos,felipe.santos@email.com,Recife,PE,Monitor 24,Eletrônicos,1.0,1099.0,Cartão De Débito,concluido
...,...,...,...,...,...,...,...,...,...,...,...,...
267,v0282,2025-11-21,Patrícia Nunes,patricia.nunes@email.com,Fortaleza,CE,Teclado Mecânico,Acessórios,3.0,389.5,Boleto,concluído
268,v0235,2025-11-22,Felipe Santos,felipe.santos@email.com,Recife,PE,Webcam,Eletrônicos,1.0,319.0,Cartão De Débito,concluído
269,v0188,2025-11-23,Sofia Cardoso,sofia.cardoso@email.com,Salvador,BA,Cadeira Ergonômica,Móveis,4.0,1490.0,Pix,pendente
270,v0141,2025-11-24,Henrique Martins,henrique.martins@email.com,Curitiba,PR,Mouse Sem Fio,Acessórios,2.0,129.9,Cartão De Crédito,concluído


In [88]:
# 8. Executar testes de qualidade e exportar sem o índice.
print("TESTES DE QUALIDADE:")
print(f"Total de registros: {len(df)}")
print(f"Registros esperados: 272")
print(f"Valores nulos por coluna:")
print(df.isnull().sum())
print(f"\nDuplicatas: {df.duplicated().sum()}")
print(f"E-mails válidos: {df['email'].apply(validar_email).sum()} de {len(df)}")
print(f"Preços positivos: {(df['preco_unitario'] > 0).sum()} de {len(df)}")
print(f"Quantidades positivas: {(df['quantidade'] > 0).sum()} de {len(df)}")

df.to_csv('vendas_tratadas.csv', sep=';', index=False, encoding='utf-8-sig')
print("\nArquivo exportado como 'vendas_tratadas.csv'")


TESTES DE QUALIDADE:
Total de registros: 272
Registros esperados: 272
Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
email              0
cidade             0
uf                 0
produto            0
categoria          0
quantidade         0
preco_unitario     0
forma_pagamento    0
status_pedido      0
dtype: int64

Duplicatas: 0
E-mails válidos: 272 de 272
Preços positivos: 272 de 272
Quantidades positivas: 272 de 272

Arquivo exportado como 'vendas_tratadas.csv'
